# 01 — Web Scraping: Papal Encyclicals Corpus

**DS 5001 — Exploratory Text Analytics Final Project**  
**Source:** https://www.papalencyclicals.net/document-directory

This notebook walks through scraping the papal encyclicals corpus from
papalencyclicals.net. The site organizes documents by pope, with each
encyclical linked to a page containing the full text (usually HTML, sometimes epub).

## Steps
1. Scrape the document directory to build an index of all encyclicals
2. Download the full text of each encyclical
3. Save raw text files and metadata

In [ ]:
import sys
sys.path.insert(0, '..')

from src.scraper import (
    scrape_index, scrape_documents, save_index, save_library_csv,
    fetch_page, parse_directory, INDEX_FILE, RAW_DIR, DATA_DIR
)
import json
import pandas as pd
from pathlib import Path

## Step 1: Scrape the Document Directory

The directory page lists encyclicals organized by pope. We parse it to
extract document titles, URLs, and associated pope names.

In [ ]:
# Scrape the directory index (or load if already scraped)
if INDEX_FILE.exists():
    with open(INDEX_FILE) as f:
        documents = json.load(f)
    print(f"Loaded existing index: {len(documents)} documents")
else:
    documents = scrape_index()
    save_index(documents)
    print(f"Scraped index: {len(documents)} documents")

In [ ]:
# Preview the index
df_index = pd.DataFrame(documents)
print(f"Documents per pope:")
print(df_index['pope'].value_counts().head(20))
print(f"\nYears covered: {df_index['year'].min()} — {df_index['year'].max()}")

## Step 2: Download Document Texts

For each document in the index, we fetch the linked page and extract
the encyclical text. The scraper handles:
- HTML pages with text in the page body
- Pages that link to epub files
- Language detection (English, Latin, Italian, French)

In [ ]:
# Scrape all document texts (skips already-downloaded files)
# Use max_docs to limit for testing:
# documents = scrape_documents(documents, max_docs=10)

documents = scrape_documents(documents)
save_index(documents)
save_library_csv(documents)

In [ ]:
# Summary
df = pd.DataFrame(documents)
print(f"Total documents: {len(df)}")
print(f"\nBy language:")
print(df['language'].value_counts())
print(f"\nBy format:")
print(df['format'].value_counts())
print(f"\nDocuments with text: {(df['text_length'] > 100).sum()}")

In [ ]:
# List raw text files
raw_files = list(RAW_DIR.glob('*.txt'))
print(f"Raw text files on disk: {len(raw_files)}")
sizes = [f.stat().st_size for f in raw_files]
print(f"Total size: {sum(sizes)/1024/1024:.1f} MB")
print(f"Average size: {sum(sizes)/len(sizes)/1024:.1f} KB")